# 07 — Policy options appraisal

**Question:** which policy levers would most improve the chance that CNG conversion pays for Lagos commercial
drivers, and which are worth doing first once cost, speed, feasibility and evidence are counted?

**Two parts:**
1. **Simulated impact.** Each option re-runs the Monte Carlo from notebook 06 (`src/policy_options.py`) with one
   input changed. All runs use the same seed and the same draws, so the difference from baseline is the policy's
   effect alone.
2. **Multi-criteria appraisal.** `docs/policy_options.csv` scores each option 1–5 (5 = best) on impact, cost, speed,
   feasibility and evidence, with a rationale for every score. Each rationale is tagged:
   - **[Simulated]**: from the model (impact only).
   - **[Calculated]**: from sourced inputs.
   - **[Judgement]**: the researcher's assessment, with sources where they exist.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
import policy_options as po   # noqa: E402

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#ecebe7"
BLUE, GREY = "#2a78d6", "#8a8984"
pd.options.display.max_colwidth = None

## 2. Simulated impact of each option

10,000 draws, seed 42, market-price conversion (except option 3). "Change" is in **percentage points** (pts): going
from 62% to 92% is +30 pts.

In [2]:
impact = po.run_options()
# Baseline is the reference point, so it gets no impact score of its own.
impact["impact_score"] = impact["change_pts"].apply(po.impact_score).where(impact.index != "baseline")
impact.style.format({"p_pays": "{:.1%}", "median_net_benefit": "₦{:,.0f}", "change_pts": "{:+.1f}",
                     "change_median": "₦{:+,.0f}", "impact_score": "{:.0f}"}, na_rep="–")

,description,p_pays,median_net_benefit,change_pts,change_median,impact_score
option,,,,,,
baseline,As surveyed: market-price conversion on current loan terms,61.8%,"₦4,440",+0.0,₦+0,–
option 1,More refuelling points: daily queue time halved,91.5%,"₦14,426",+29.7,"₦+9,986",4
option 2,More capacity at existing stations: daily queue capped at 1 hour,98.2%,"₦15,125",+36.4,"₦+10,685",5
option 3,Free conversion,65.5%,"₦6,141",+3.8,"₦+1,701",2
option 4,Cheaper loans: interest fixed at 5%,62.3%,"₦4,640",+0.5,₦+200,1
option 5,"Stable CNG price: fixed at the mid value, NGN 318/scm",61.5%,"₦4,238",-0.2,₦-202,1


## 3. The scoring table

Loaded from `docs/policy_options.csv`. The first check confirms that the CSV's impact scores still match a fresh
simulation, so the CSV can't silently go out of date if the model or inputs change.

In [3]:
scores = pd.read_csv(PROJECT_ROOT / "docs" / "policy_options.csv").set_index("option")
CRITERIA = ["impact", "cost", "speed", "feasibility", "evidence"]

fresh = impact.loc[scores.index, "impact_score"]
assert (scores["impact_score"] == fresh).all(), "policy_options.csv impact scores are out of date - regenerate them"
assert scores[[f"{c}_score" for c in CRITERIA]].isin(range(1, 6)).all().all(), "scores must be 1-5"

scores[["description", "lead_agency"] + [f"{c}_score" for c in CRITERIA]]

,description,lead_agency,impact_score,cost_score,speed_score,feasibility_score,evidence_score
option,,,,,,,
option 1,More refuelling points: new stations or mobile units so daily queue time halves,"Pi-CNG, with NMDPRA (licensing) and Lagos State / IBILE (IOGC)",4,2,2,3,4
option 2,"More capacity at existing stations: compressors, dispensers or opening hours so no daily queue exceeds 1 hour","Station operators (NNPC Retail / NIPCO, IBILE), supported by Pi-CNG and MDGIF",5,3,4,3,3
option 3,Free conversion: government pays for the kit and installation,Pi-CNG,2,1,4,4,3
option 4,Cheaper loans: conversion loans at 5% a year instead of 15-20%,MOFI / CREDICORP,1,3,4,4,2
option 5,Stable CNG price: hold the pump price at the mid value instead of letting it vary,"NMDPRA, with NADDC",1,3,3,2,2


### Rationale for every score

In [4]:
# One row per option x criterion, so each rationale is readable on its own line.
rationales = pd.concat(
    [scores[[f"{c}_score", f"{c}_rationale"]].set_axis(["score", "rationale"], axis=1).assign(criterion=c)
     for c in CRITERIA]
).reset_index().set_index(["option", "criterion"]).sort_index(level=0, sort_remaining=False)
rationales = rationales.reindex(CRITERIA, level=1)
rationales.style.set_properties(subset=["rationale"], **{"text-align": "left", "white-space": "normal"})

## 4. Weighted scores and ranking

Weights: **impact 35%, cost 20%, speed 15%, feasibility 15%, evidence 15%**. Weighted score =
Σ weight × score, so it is also on a 1–5 scale.

In [5]:
WEIGHTS = {"impact": 0.35, "cost": 0.20, "speed": 0.15, "feasibility": 0.15, "evidence": 0.15}


def weighted(weights):
    assert abs(sum(weights.values()) - 1) < 1e-9, "weights must add up to 1"
    # Multiply each score column by its weight and add across the row.
    return sum(scores[f"{c}_score"] * w for c, w in weights.items())


ranking = scores[["description", "lead_agency"]].copy()
ranking["weighted_score"] = weighted(WEIGHTS)
ranking["rank"] = ranking["weighted_score"].rank(ascending=False, method="min").astype(int)
ranking = ranking.sort_values("rank")
ranking.style.format({"weighted_score": "{:.2f}"})

,description,lead_agency,weighted_score,rank
option,,,,
option 2,"More capacity at existing stations: compressors, dispensers or opening hours so no daily queue exceeds 1 hour","Station operators (NNPC Retail / NIPCO, IBILE), supported by Pi-CNG and MDGIF",3.85,1
option 1,More refuelling points: new stations or mobile units so daily queue time halves,"Pi-CNG, with NMDPRA (licensing) and Lagos State / IBILE (IOGC)",3.15,2
option 3,Free conversion: government pays for the kit and installation,Pi-CNG,2.55,3
option 4,Cheaper loans: conversion loans at 5% a year instead of 15-20%,MOFI / CREDICORP,2.45,4
option 5,Stable CNG price: hold the pump price at the mid value instead of letting it vary,"NMDPRA, with NADDC",2.00,5


## 5. Robustness check: do the weights change the answer?

Two alternative weightings:
- **Equal:** 20% each.
- **Impact 50%:** impact gets 50%. The other four share the remaining 50% in the same proportions as before
  (20 : 15 : 15 : 15).

In [6]:
others = {c: w for c, w in WEIGHTS.items() if c != "impact"}
weight_sets = {
    "base (35% impact)": WEIGHTS,
    "equal (20% each)": {c: 0.2 for c in CRITERIA},
    "impact 50%": {"impact": 0.5, **{c: 0.5 * w / sum(others.values()) for c, w in others.items()}},
}
robust = pd.DataFrame({name: weighted(w) for name, w in weight_sets.items()})
ranks = robust.rank(ascending=False, method="min").astype(int).add_suffix(" rank")
robust = pd.concat([robust, ranks], axis=1).sort_values("base (35% impact) rank")

top_two = {name: list(robust[f"{name} rank"].sort_values().index[:2]) for name in weight_sets}
for name, top in top_two.items():
    print(f"{name:18s} top two: {top}")
base_top = set(top_two["base (35% impact)"])
same = all(set(t) == base_top for t in top_two.values())
print("\nTop two unchanged under every weighting." if same else "\nTop two CHANGE under at least one weighting.")
robust.style.format({c: "{:.2f}" for c in weight_sets})

base (35% impact)  top two: ['option 2', 'option 1']
equal (20% each)   top two: ['option 2', 'option 1']
impact 50%         top two: ['option 2', 'option 1']

Top two unchanged under every weighting.


,base (35% impact),equal (20% each),impact 50%,base (35% impact) rank,equal (20% each) rank,impact 50% rank
option,,,,,,
option 2,3.85,3.60,4.12,1,1,1
option 1,3.15,3.00,3.35,2,2,2
option 3,2.55,2.80,2.42,3,3,3
option 4,2.45,2.80,2.12,4,3,4
option 5,2.00,2.20,1.77,5,5,5


## 6. Chart: probability CNG pays, by option

Design choices: a single measure, so one colour for the options and grey for the baseline, which is also marked
with a dashed line so each bar can be compared against it. Each bar is labelled with its probability and its change
from baseline.

In [7]:
SHORT = {
    "baseline": "Baseline (as surveyed)",
    "option 1": "1. More refuelling points (queues halved)",
    "option 2": "2. More capacity (queue capped at 1 h/day)",
    "option 3": "3. Free conversion",
    "option 4": "4. Cheaper loans (5% interest)",
    "option 5": "5. Stable CNG price (fixed at mid)",
}
plot = impact.iloc[::-1]   # plotly draws the first bar at the bottom; keep baseline on top
base_p = impact.loc["baseline", "p_pays"]
labels = [f"{p:.0%}" if o == "baseline" else f"{p:.0%}  ({c:+.1f} pts)"
          for o, p, c in zip(plot.index, plot["p_pays"], plot["change_pts"])]

fig = go.Figure(go.Bar(
    x=plot["p_pays"], y=[SHORT[o] for o in plot.index], orientation="h", width=0.6,
    marker=dict(color=[GREY if o == "baseline" else BLUE for o in plot.index], cornerradius=4),
    text=labels, textposition="outside", cliponaxis=False, textfont=dict(color=INK, size=12),
    customdata=plot[["median_net_benefit"]],
    hovertemplate="<b>%{y}</b><br>P(CNG pays): %{x:.1%}<br>Median net benefit ₦%{customdata[0]:,.0f}/day"
                  "<extra></extra>",
))
fig.add_vline(x=base_p, line=dict(color=INK_MUTED, width=1.5, dash="dash"))
fig.add_annotation(x=base_p, y=1, yref="paper", yanchor="bottom", showarrow=False,
                   text=f"Baseline {base_p:.0%}", font=dict(color=INK_MUTED, size=12))
fig.update_layout(
    title=dict(text="Cutting queues raises the chance CNG pays far more than cheaper kits, loans or prices<br>"
                    "<sup>Probability net benefit > 0, 10,000 Monte Carlo draws per option (seed 42), "
                    "market-price conversion except option 3. Survey drivers (S47).</sup>", x=0, xanchor="left"),
    template="plotly_white", font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color=INK),
    xaxis=dict(title="Probability CNG pays", tickformat=".0%", range=[0, 1.08], dtick=0.2, gridcolor=GRID,
               zeroline=False),
    yaxis=dict(ticksuffix="  "),
    hoverlabel=dict(bgcolor="white", font=dict(color=INK)), showlegend=False,
    width=860, height=440, margin=dict(t=110, l=20, r=40, b=60),
)
fig.show()
fig.write_html(FIG_DIR / "policy_options_impact.html", include_plotlyjs="cdn")
print("Saved outputs/figures/policy_options_impact.html")

Saved outputs/figures/policy_options_impact.html


## 7. What the ranking means

**The two queue options come first, under every weighting tested.** More capacity at existing stations (option 2,
weighted score 3.85) ranks first and more refuelling points (option 1, 3.15) second. They stay first and second
with equal weights and with impact weighted at 50%. Free conversion (2.55), cheaper loans (2.45) and a stable CNG
price (2.00) follow. Cheaper loans tie with free conversion for third under equal weights.

**Why:** in the simulation, queue time is what decides whether CNG pays (notebook 06). Halving daily queues raises the
probability that CNG pays from **62% to 92%** (+30 pts); capping them at one hour raises it to **98%** (+36 pts).
Both cut average daily queuing to about an hour (0.99 and 0.92 hours, from 1.98). The cap scores higher because it
also removes the longest queues, which are the ones that turn a gain into a loss. By contrast, a free conversion adds
only about 4 points, a 5% loan under 1 point, and fixing the CNG price at its mid value nothing at all. For
drivers like those surveyed, the loan repayment is small next to fuel savings and queue costs, so making kits
cheaper changes little.

**Option 1 sits just below the top impact band** (+29.7 pts against a threshold of 30). Scoring it 5 instead of 4
would raise its weighted score to 3.50, still second.

**Recommendations this supports:**
1. **Raise capacity at the busiest existing stations first.** Add compressors and dispensers, extend opening hours,
   and send mobile units to sites with the longest queues. The aim is that no driver queues more than about an hour a
   day. *Lead: station operators (NNPC Retail / NIPCO, IBILE), supported by Pi-CNG and MDGIF.* It is the fastest of
   the queue options because it builds on sites that already have land and a gas connection.
2. **Add refuelling points where there are none confirmed.** Notebook 05 found only 6 confirmed refuelling points,
   and none in Alimosho, Ikorodu, Badagry, Ibeju-Lekki or Lagos Island. Confirming whether the commissioned IBILE sites
   are actually selling gas is a cheap first step. *Lead: Pi-CNG, with NMDPRA (licensing) and Lagos State / IBILE.*
   This option is slower and more expensive, so it is a medium-term programme rather than a quick fix.
3. **Tie any expansion of free conversion to station capacity.** Free kits help a little on their own, at a cost of
   about ₦130 bn per 100,000 car conversions (S40). More CNG vehicles without more station capacity means longer
   queues, which is what this analysis finds matters most. Delhi's early CNG queues came with a 32% gas supply
   deficit (S27). *Lead: Pi-CNG.*
4. **Don't prioritise cheaper loans or fixed CNG prices on this evidence.** Neither moves the probability
   meaningfully in the model. Cutting the loan rate from 17.5% to 5% gives up about ₦181,000 of interest per car loan
   (S40, S18). Holding prices risks supply. A published price band could still help drivers plan, but that benefit is
   outside what the model measures. *Lead if pursued: MOFI / CREDICORP (loans); NMDPRA with NADDC (prices).*

**Limitations:**
- **Scores are partly judgement.** Only the impact score comes from the simulation, and only option 3's cost (and
  part of option 4's) is calculated. Every other cost, speed, feasibility and evidence score is the researcher's
  judgement, marked [Judgement] in `docs/policy_options.csv` with sources where they exist. Different judgements could
  change the order below the top two.
- **Options are modelled separately, not combined.** Each run changes one input. In practice the options interact:
  free conversion would bring more vehicles to the same stations and lengthen queues, which the model does not
  capture.
- **Policy sizes are choices.** "Queues halved", "capped at 1 hour" and "5% interest" are test settings, not
  costed programmes. There is no evidence here on how much investment would deliver them.
- **Costs are indicative.** The ₦130 bn figure uses a single car-kit price (S40) and ignores keke kits, installation
  capacity and administration. No sourced naira cost was found for new stations or capacity upgrades, so those are
  rated Low/Medium/High only.
- **Everything inherits the Monte Carlo's limitations** (notebook 06):
  - The simulated drivers are 20 surveyed drivers, re-sampled.
  - The input distributions are assumed triangles.
  - Inputs are drawn independently of each other.
  - Earnings are before owner remittance or app commission.